## optional if errored

Fixes `CERTIFICATE_VERIFY_FAILED` on a TLS-inspecting corporate network. Run before any other cell, and after every kernel restart.


In [7]:
%pip install -q truststore

import ssl
import truststore

truststore.inject_into_ssl()

with ssl.create_default_context().wrap_socket(
    __import__("socket").create_connection(("api.kaggle.com", 443), timeout=15),
    server_hostname="api.kaggle.com",
) as sock:
    print("TLS handshake with api.kaggle.com OK:", sock.version())



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: C:\Users\CE977RH\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
TLS handshake with api.kaggle.com OK: TLSv1.3


In [11]:
import kagglehub

kagglehub.login()

In [12]:
import logging
import time
from pathlib import Path

import kagglehub
import kagglehub.clients as kh_clients
import requests
from kagglehub.cache import get_cached_archive_path
from kagglehub.handle import parse_dataset_handle

HANDLE = "rajat95gupta/smartphone-image-denoising-dataset"
MAX_ATTEMPTS = 25

# Read timeout is the gap allowed *between* 1 MB chunks; kagglehub's default of 15s is too
# tight for a throttling proxy and aborts the resumed transfer.
kh_clients.DEFAULT_CONNECT_TIMEOUT = 30
kh_clients.DEFAULT_READ_TIMEOUT = 300

# kagglehub reports "Resuming download from N bytes" at INFO level.
logging.basicConfig(level=logging.INFO, force=True)

# The archive is named after the dataset version, which isn't known until the first request.
ARCHIVE_DIR = Path(get_cached_archive_path(parse_dataset_handle(HANDLE))).parent


def staged_bytes() -> int:
    return sum(f.stat().st_size for f in ARCHIVE_DIR.glob("*.archive"))


RETRYABLE = (
    requests.exceptions.ChunkedEncodingError,
    requests.exceptions.ConnectionError,
    requests.exceptions.Timeout,
    ConnectionResetError,
)

print(f"Staging into {ARCHIVE_DIR} ({staged_bytes() / 1e9:.2f} GB already present)")

path = None
for attempt in range(1, MAX_ATTEMPTS + 1):
    before = staged_bytes()
    try:
        path = kagglehub.dataset_download(HANDLE)
        break
    except RETRYABLE as exc:
        backoff = min(60, 2**attempt)
        gained = (staged_bytes() - before) / 1e9
        print(
            f"[attempt {attempt}/{MAX_ATTEMPTS}] {type(exc).__name__}: +{gained:.2f} GB this attempt, "
            f"{staged_bytes() / 1e9:.2f} GB staged total; resuming in {backoff}s",
            flush=True,
        )
        time.sleep(backoff)
else:
    raise RuntimeError(f"Download still incomplete after {MAX_ATTEMPTS} attempts; {staged_bytes() / 1e9:.2f} GB staged")

print("Path to dataset files:", path)


Staging into C:\Users\CE977RH\.cache\kagglehub\datasets\rajat95gupta\smartphone-image-denoising-dataset (1.39 GB already present)
Resuming download from 1394606080 bytes (5222928497 bytes left)...
Resuming download to C:\Users\CE977RH\.cache\kagglehub\datasets\rajat95gupta\smartphone-image-denoising-dataset\1.archive (1394606080/6617534577) bytes left.


 27%|██▋       | 1.66G/6.16G [12:56<2:39:25, 505kB/s] 

[attempt 1/25] ChunkedEncodingError: +0.39 GB this attempt, 1.79 GB staged total; resuming in 2s


[attempt 2/25] ConnectionError: +0.00 GB this attempt, 1.79 GB staged total; resuming in 4s
[attempt 3/25] ConnectionError: +0.00 GB this attempt, 1.79 GB staged total; resuming in 8s
Resuming download from 1786773504 bytes (4830761073 bytes left)...
Resuming download to C:\Users\CE977RH\.cache\kagglehub\datasets\rajat95gupta\smartphone-image-denoising-dataset\1.archive (1786773504/6617534577) bytes left.


 27%|██▋       | 1.67G/6.16G [05:35<429:42:42, 3.12kB/s]

...

[attempt 19/25] ConnectionError: +0.14 GB this attempt, 3.57 GB staged total; resuming in 60s


Resuming download from 3574595584 bytes (3042938993 bytes left)...
Resuming download to C:\Users\CE977RH\.cache\kagglehub\datasets\rajat95gupta\smartphone-image-denoising-dataset\1.archive (3574595584/6617534577) bytes left.


100%|██████████| 6.16G/6.16G [18:03<00:00, 2.81MB/s] 

Extracting files...


Path to dataset files: C:\Users\CE977RH\.cache\kagglehub\datasets\rajat95gupta\smartphone-image-denoising-dataset\versions\1


In [13]:
SIDD_DIR = Path(path) / "SIDD_Small_sRGB_Only" / "Data"
scenes = sorted(p for p in SIDD_DIR.iterdir() if p.is_dir())

print(f"SIDD_DIR: {SIDD_DIR}")
print(f"scenes:   {len(scenes)}")

pairs = [
    (next(s.glob("NOISY_SRGB_*.PNG")), next(s.glob("GT_SRGB_*.PNG")))
    for s in scenes
]
print(f"pairs:    {len(pairs)}")
print(f"example:  {pairs[0][0].relative_to(SIDD_DIR)} <-> {pairs[0][1].name}")


SIDD_DIR: C:\Users\CE977RH\.cache\kagglehub\datasets\rajat95gupta\smartphone-image-denoising-dataset\versions\1\SIDD_Small_sRGB_Only\Data
scenes:   160
pairs:    160
example:  0001_001_S6_00100_00060_3200_L\NOISY_SRGB_010.PNG <-> GT_SRGB_010.PNG
